In [ ]:
import numpy as np
from IPython.display import clear_output
import gc

# After heavy computations
clear_output(wait=True)
gc.collect()

In [ ]:
# Start to analyze the channel result
import os, re, json, glob
from pathlib import Path
import pandas as pd

# Root where results live (change if needed)
RESULTS_ROOT = Path("result")

# Glob pattern for metrics files
GLOB_PATTERN = str(RESULTS_ROOT / "P*" / "Seizure*" / "analysis_results" / "metrics_*.json")

# Regex to extract patient/seizure numbers from the path
PAT_RE = re.compile(r"P(\d+)")
SEIZ_RE = re.compile(r"Seizure(\d+)")

rows = []

for fp in glob.iglob(GLOB_PATTERN, recursive=True):
    p = Path(fp)
    try:
        # Derive patient/seizure from folder names
        m_pat = PAT_RE.search(str(p))
        m_seiz = SEIZ_RE.search(str(p))
        patient = int(m_pat.group(1)) if m_pat else None
        seizure = int(m_seiz.group(1)) if m_seiz else None

        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)

        perf = data.get("performance", {}) or {}
        params = data.get("parameters", {}) or {}
        acc_all = perf.get("accuracy_all_channels", None)
        model_name = data.get("model_name") or "Unknown"
        ts = data.get("timestamp") or ""

        rows.append({
            "patient": patient,
            "seizure": seizure,
            "model": model_name,
            "accuracy_all_channels": acc_all,
            "timestamp": ts,
            "file": str(fp),
            # (optional) keep a few knobs that often matter
            "threshold": params.get("threshold"),
            "smooth_window": params.get("smooth_window"),
            "n_seconds": params.get("n_seconds"),
            "overlap": params.get("overlap"),
        })
    except Exception as e:
        # Keep going even if one file is malformed
        rows.append({
            "patient": None, "seizure": None, "model": "ERROR",
            "accuracy_all_channels": None, "timestamp": "", "file": str(fp),
            "threshold": None, "smooth_window": None, "n_seconds": None, "overlap": None,
            "error": str(e),
        })

df = pd.DataFrame(rows)

# Keep only the latest record per (patient, seizure, model) using 'timestamp'
# Assumes timestamp sorts lexicographically (e.g., "YYYYMMDD_HHMMSS")
if not df.empty:
    df["timestamp"] = df["timestamp"].fillna("")
    df_sorted = df.sort_values(["patient", "seizure", "model", "timestamp"])
    # Drop duplicates keeping the last (latest)
    df_latest = df_sorted.drop_duplicates(subset=["patient", "seizure", "model"], keep="last")
else:
    df_latest = df

# Save summaries
out_dir = RESULTS_ROOT
out_dir.mkdir(parents=True, exist_ok=True)
csv_all = out_dir / "seizure_accuracy_all_rows.csv"
csv_latest = out_dir / "seizure_accuracy_latest_per_model.csv"

df.to_csv(csv_all, index=False, encoding="utf-8-sig")
df_latest.to_csv(csv_latest, index=False, encoding="utf-8-sig")

print(f"Saved all rows to: {csv_all}")
print(f"Saved latest-per-(patient,seizure,model) to: {csv_latest}")

# (Optional) quick pivot: best accuracy per seizure across models
if not df_latest.empty:
    pivot = df_latest.pivot_table(
        index=["patient", "seizure"],
        columns="model",
        values="accuracy_all_channels",
        aggfunc="max"
    )
    pivot_path = out_dir / "seizure_accuracy_pivot.csv"
    pivot.to_csv(pivot_path, encoding="utf-8-sig")
    print(f"Saved pivot to: {pivot_path}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import itertools
from scipy import stats

# ---- 1) rename long model names to short ones ----
name_map = {
    "CustomTransformerSeizurePredictor": "Transformer",
    "EnhancedResNet": "ResNet",
    "ResNet": "ResNet",
    "Wavenet": "Wavenet",
    "WaveNet": "Wavenet",
}
df = df.copy()
df["model"] = df["model"].map(lambda x: name_map.get(x, x))

# Optional: fix plotting order
model_order = ["Transformer", "ResNet", "Wavenet"]
df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)

sns.set_theme(style="whitegrid", context="talk")

# ---- 2) facet per patient; x=model; y=accuracy ----
g = sns.FacetGrid(
    df, col="patient", col_wrap=2, sharey=True,
    height=4.6, aspect=1.15, margin_titles=True
)

def draw_facet(data, **kwargs):
    ax = plt.gca()
    # box + points
    sns.boxplot(data=data, x="model", y="accuracy_all_channels",
                order=model_order, width=0.6, ax=ax)
    sns.stripplot(data=data, x="model", y="accuracy_all_channels",
                  order=model_order, color="k", size=4, alpha=0.6, jitter=True, ax=ax)
    ax.set_ylim(0.35, 1.05)
    ax.set_xlabel("Model")
    ax.set_ylabel("Accuracy (all channels)")
    # rotate & right-align x labels
    for tick in ax.get_xticklabels():
        tick.set_rotation(20)
        tick.set_ha("right")

    # (optional) simple pairwise stats + asterisks (no external lib)
    models = [m for m in model_order if m in data["model"].unique()]
    pairs = list(itertools.combinations(models, 2))
    if not pairs:
        return
    # collect p-values (Welch by default; fallback if tiny N)
    pvals = []
    for m1, m2 in pairs:
        a = data.loc[data["model"]==m1, "accuracy_all_channels"].to_numpy(float)
        b = data.loc[data["model"]==m2, "accuracy_all_channels"].to_numpy(float)
        p = stats.ttest_ind(a, b, equal_var=False).pvalue if (len(a)>=2 and len(b)>=2) \
            else stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        pvals.append(p)
    # BH-FDR within facet
    p = np.array(pvals, float)
    order = np.argsort(p); ranks = np.arange(1, len(p)+1)
    adj = np.empty_like(p); adj[order] = np.minimum.accumulate((p[order]*len(p)/ranks)[::-1])[::-1]
    adj = np.clip(adj, 0, 1)

    def stars(x):
        return "***" if x<1e-3 else "**" if x<1e-2 else "*" if x<5e-2 else "ns"

    # x positions of categories
    xticks = ax.get_xticks()
    xlabels = [t.get_text() for t in ax.get_xticklabels()]
    xpos = {lab: xticks[i] for i, lab in enumerate(xlabels)}
    # draw compact bars; increase top margin first
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    y = ymax + 0.03*span
    for (m1, m2), p_adj in zip(pairs, adj):
        x1, x2 = xpos[m1], xpos[m2]
        if x1 > x2: x1, x2 = x2, x1
        ax.plot([x1, x1, x2, x2], [y, y+0.01*span, y+0.01*span, y], color="crimson", lw=1.5)
        ax.text((x1+x2)/2, y+0.012*span, stars(p_adj), color="crimson",
                ha="center", va="bottom", fontsize=12, fontweight="bold")
        y += 0.05*span  # stack bars

    ax.set_ylim(ymin, y + 0.02*span)  # give headroom for bars

g.map_dataframe(draw_facet)

# more room for facet titles and suptitle
g.set_titles("patient = {col_name}")
# g.figure.subplots_adjust(top=0.88, bottom=0.12)
g.figure.suptitle("Prediction Accuracy per Patient (model comparisons with FDR)", fontsize=16)
plt.savefig(out_dir / "seizure_accuracy_per_patient.png", dpi=150)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import itertools
from scipy import stats

# df: patient, model, accuracy_all_channels
# e.g. df = pd.read_csv("result/seizure_accuracy_latest_per_model.csv")
df = df[['patient','model','accuracy_all_channels']].dropna()
df['accuracy_all_channels'] = df['accuracy_all_channels'].astype(float)

# optional: shorten long model names
name_map = {
    "CustomTransformerSeizurePredictor": "Transformer",
    "EnhancedResNet": "ResNet",
    "ResNet": "ResNet",
    "Wavenet": "Wavenet",
    "WaveNet": "Wavenet",
}
df['model'] = df['model'].map(lambda x: name_map.get(x, x))
model_order = ["Transformer","ResNet","Wavenet"]
df['model'] = pd.Categorical(df['model'], categories=model_order, ordered=True)

sns.set_theme(style="whitegrid", context="talk")

plt.figure(figsize=(8,6))
ax = sns.boxplot(
    data=df, x="model", y="accuracy_all_channels",
    order=model_order, hue="patient", width=0.6
)
sns.stripplot(
    data=df, x="model", y="accuracy_all_channels",
    order=model_order, hue="patient", dodge=True,
    marker="o", alpha=0.6, size=5, ax=ax
)

# remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(df['patient'].unique())],
          labels[:len(df['patient'].unique())],
          title="Patient", bbox_to_anchor=(1.05,1), loc="upper left")

ax.set_ylim(0.35,1.1)
ax.set_ylabel("Accuracy (all channels)")
ax.set_xlabel("Model")
ax.set_title("Prediction Accuracy across Patients and Models")

# --- statistics across patients per model (global test between models) ---
# group by model, pool all patients together
groups = [df.loc[df['model']==m, 'accuracy_all_channels'] for m in model_order]
pairs = list(itertools.combinations(range(len(model_order)),2))

# compute p-values (Welch t-test)
pvals = []
for i,j in pairs:
    a, b = groups[i], groups[j]
    p = stats.ttest_ind(a, b, equal_var=False).pvalue if (len(a)>=2 and len(b)>=2) \
        else stats.mannwhitneyu(a,b,alternative="two-sided").pvalue
    pvals.append(p)

# Benjamini-Hochberg correction
p = np.array(pvals); n=len(p)
order = np.argsort(p); ranks=np.arange(1,n+1)
adj = np.empty_like(p)
adj[order] = np.minimum.accumulate((p[order]*n/ranks)[::-1])[::-1]
adj = np.clip(adj,0,1)


plt.tight_layout()
plt.savefig(out_dir / "seizure_accuracy_across_patients.png", dpi=150)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import itertools
from scipy import stats

# df must have: model, accuracy_all_channels  (patient column can be present; it will be ignored/pooled)
# Example: df = pd.read_csv("result/seizure_accuracy_latest_per_model.csv")
df = df[['model','accuracy_all_channels']].dropna()
df['accuracy_all_channels'] = df['accuracy_all_channels'].astype(float)

# Optional: shorten long model names and fix plotting order
name_map = {
    "CustomTransformerSeizurePredictor": "Transformer",
    "EnhancedResNet": "ResNet",
    "ResNet": "ResNet",
    "Wavenet": "Wavenet", "WaveNet": "Wavenet",
}

df['model'] = df['model'].map(lambda x: name_map.get(x, x))
model_order = ["Transformer", "ResNet", "Wavenet"]
df['model'] = pd.Categorical(df['model'], categories=model_order, ordered=True)

sns.set_theme(style="whitegrid", context="talk")
plt.figure(figsize=(7.5, 5.5))
ax = sns.boxplot(
    data=df, x="model", y="accuracy_all_channels",
    order=model_order, width=0.6
)
sns.stripplot(
    data=df, x="model", y="accuracy_all_channels",
    order=model_order, color="k", size=4, alpha=0.6, jitter=True, ax=ax
)
ax.set_ylim(0.35, 1.05)
ax.set_xlabel("Model")
ax.set_ylabel("Accuracy (all channels)")
ax.set_title("Pooled accuracy across all patients")

# ---------- Pairwise stats across models (pooled) ----------
groups = [df.loc[df['model']==m, 'accuracy_all_channels'].values for m in model_order]
pairs = list(itertools.combinations(range(len(model_order)), 2))

# p-values (Welch t-test; fallback to Mann–Whitney for tiny N)
pvals = []
for i, j in pairs:
    a, b = groups[i], groups[j]
    if len(a) >= 2 and len(b) >= 2:
        p = stats.ttest_ind(a, b, equal_var=False).pvalue
    else:
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    pvals.append(p)

# Benjamini–Hochberg FDR
p = np.asarray(pvals, float)
if len(p) > 0:
    order = np.argsort(p); ranks = np.arange(1, len(p)+1)
    adj = np.empty_like(p)
    adj[order] = np.minimum.accumulate((p[order] * len(p) / ranks)[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
else:
    adj = p

def stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else "ns"

# Draw significance bars
ymin, ymax = ax.get_ylim()
span = ymax - ymin
y = ymax + 0.04 * span
for (i, j), p_adj in zip(pairs, adj):
    x1, x2 = i, j
    ax.plot([x1, x1, x2, x2], [y, y+0.01*span, y+0.01*span, y], color="crimson", lw=1.5)
    ax.text((x1+x2)/2, y + 0.012*span, stars(p_adj),
            ha="center", va="bottom", fontsize=12, fontweight="bold", color="crimson")
    y += 0.06 * span  # stack bars

ax.set_ylim(ymin, y + 0.02*span)
plt.tight_layout()
plt.savefig(out_dir / "seizure_accuracy_pooled.png", dpi=150)
plt.show()
